# 👥 View — Clientes por Região

Validação da view `vw_clientes_regiao` antes de mover para o Streamlit.

In [1]:
import pandas as pd
import sys
sys.path.append('..')

pd.set_option('display.float_format', '{:.2f}'.format)

pedidos  = pd.read_csv("../dados/pedidos_limpo.csv", parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
clientes = pd.read_csv("../dados/clientes_limpo.csv")

print("Dados carregados!")

Dados carregados!


## 🧪 Testando o código antes de criar a view

In [2]:
pedidos.groupby('order_status')['order_id'].nunique().sort_values(ascending=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: order_id, dtype: int64

In [3]:
pedidos['order_id'].nunique()

99441

In [4]:
pedidos.groupby('order_status').size().sort_values(ascending=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
dtype: int64

In [ ]:
# Join entre pedidos e clientes
df = pedidos.merge(clientes, on='customer_id', how='left')

# Extraindo ano e mês
df['ano']      = df['order_purchase_timestamp'].dt.year
df['data_mes'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Agrupando por estado, cidade e status
clientes_regiao = (df.groupby(['customer_state', 'customer_city', 'ano', 'data_mes', 'order_status'])
                     .agg(
                         total_clientes = ('customer_unique_id', 'nunique'),
                         total_pedidos  = ('order_id',           'nunique')
                     )
                     .reset_index()
                     .sort_values('total_clientes', ascending=False))

clientes_regiao.head()

In [ ]:
from views.vw_clientes_regiao import get_clientes_regiao

df_clientes = get_clientes_regiao(pedidos, clientes)
df_clientes.head()

In [ ]:
# Com sum
df_clientes.groupby(['ano', 'data_mes'])['total_clientes'].sum().reset_index().head(10)

In [ ]:
# Com max
df_clientes.groupby(['ano', 'data_mes'])['total_clientes'].max().reset_index().head(10)